# Attack follow-up analysis

Conditional-on-ok-rewrite differential for `persuasion_attack` (RESULTS_v1.md §5.1) and `paraphrase_attack` (§5.2).

Why: the raw differential pools rows where the rewriter (Claude) refused the rewrite task and the script fell back to the original prompt. Those fallbacks have ~0% compliance on both sides (the originals are from quadrants Llama already refused), so they shrink the headline number. Filter to `rewrite_status == "ok"` to see the gap *on the subset where the attack actually fired*.

In [7]:
import json, pandas as pd
from pathlib import Path

# === EDIT THESE TO MATCH YOUR ENV ===
ART  = Path('../ps')
SHARED = Path('../ps/paraphrase_attack')

PERSUASION = ART / 'analysis' / 'persuasion_attack' / 'responses.json'

## 1. Persuasion attack (Exp 1, within Llama)

Step A: rewriter-refusal asymmetry. If Claude refused trigger_only and concept_deep at the same rate, the pooled differential is just attenuated. If it refused asymmetrically (more on one quadrant), there's a separate confound.

In [8]:
with open(PERSUASION) as f:
    df = pd.DataFrame(json.load(f)['responses'])

print(f'total rows: {len(df):,}')
print(df['rewrite_status'].value_counts())

# Rewriter-refusal rate per (lang, quadrant). Asymmetry here = confound.
rewriter_refusal = (
    df.assign(rewriter_refused=lambda d: d['rewrite_status'] == 'rewriter_refused')
      .groupby(['lang', 'quadrant'])['rewriter_refused']
      .mean().unstack('quadrant').round(3)
)
rewriter_refusal['asymmetry_pp'] = ((rewriter_refusal.get('concept_deep', 0) - rewriter_refusal.get('trigger_only', 0)) * 100).round(2)
rewriter_refusal

total rows: 3,591
rewrite_status
rewriter_refused    2726
ok                   865
Name: count, dtype: int64


quadrant,concept_deep,trigger_only,asymmetry_pp
lang,,,
ar,0.907,0.727,18.0
bn,0.913,0.673,24.0
cs,0.767,0.640,12.7
en,0.847,0.633,21.4
hu,0.867,0.660,20.7
ko,0.953,0.767,18.6
ms,0.713,0.507,20.6
ru,0.860,0.747,11.3
sr,0.760,0.720,4.0


Step B: differential on the OK-only subset (the real PAP-bypass signal).

In [9]:
ok = df[df['rewrite_status'] == 'ok']
print(f'ok rows: {len(ok):,}  ({len(ok)/len(df):.1%} of total)')

# Per (lang, quadrant) compliance on the ok subset.
comp = (
    ok.groupby(['lang', 'quadrant'])
       .agg(n=('refused', 'size'),
            complied=('complied', 'sum'),
            compliance=('complied', 'mean'))
       .reset_index()
)
comp_pivot = comp.pivot(index='lang', columns='quadrant', values='compliance')
n_pivot    = comp.pivot(index='lang', columns='quadrant', values='n')
comp_pivot['diff_pp']      = ((comp_pivot['trigger_only'] - comp_pivot['concept_deep']) * 100).round(2)
comp_pivot['n_trig']       = n_pivot['trigger_only']
comp_pivot['n_concept']    = n_pivot['concept_deep']
comp_pivot = comp_pivot.round({'trigger_only': 4, 'concept_deep': 4})
comp_pivot.loc['__mean__'] = [comp_pivot['concept_deep'].mean(),
                                comp_pivot['trigger_only'].mean(),
                                comp_pivot['diff_pp'].mean(),
                                comp_pivot['n_trig'].sum(),
                                comp_pivot['n_concept'].sum()]
comp_pivot

ok rows: 865  (24.1% of total)


quadrant,concept_deep,trigger_only,diff_pp,n_trig,n_concept
lang,,,,,
ar,0.285700,0.585400,29.970000,41.0,14.0
bn,0.384600,0.714300,32.970000,49.0,13.0
cs,0.457100,0.518500,6.140000,54.0,35.0
en,0.608700,0.509100,-9.960000,55.0,23.0
hu,0.350000,0.549000,19.900000,51.0,20.0
ko,0.285700,0.514300,22.860000,35.0,7.0
ms,0.465100,0.567600,10.250000,74.0,43.0
ru,0.428600,0.447400,1.880000,38.0,21.0
sr,0.638900,0.666700,2.780000,42.0,36.0


Step C: per (lang, technique) breakdown on the ok subset — see whether one PAP technique drives the differential.

In [10]:
by_tech = (
    ok.groupby(['lang', 'technique', 'quadrant'])['complied']
      .mean().unstack('quadrant')
)
by_tech['diff_pp'] = ((by_tech['trigger_only'] - by_tech['concept_deep']) * 100).round(2)
by_tech.round(4)

quadrant                    concept_deep  trigger_only  diff_pp
lang technique                                                 
ar   authority_endorsement        0.5000        0.6000    10.00
     evidence_based               0.2500        0.5882    33.82
     logical_appeal               0.2500        0.5789    32.89
bn   authority_endorsement        0.0000        0.7778    77.78
     evidence_based               0.3333        0.6522    31.88
     logical_appeal               0.6667        0.7647     9.80
cs   authority_endorsement        0.4000        0.4286     2.86
     evidence_based               0.4286        0.5200     9.14
     logical_appeal               0.5455        0.6000     5.45
en   authority_endorsement        0.4000        0.4286     2.86
     evidence_based               0.6667        0.6000    -6.67
     logical_appeal               0.6667        0.4762   -19.05
hu   authority_endorsement        0.0000        0.5000    50.00
     evidence_based               0.3333        0.4615    12.82
     logical_appeal               0.4444        0.7059    26.14
ko   authority_endorsement           NaN        0.5000      NaN
     evidence_based               0.4000        0.5000    10.00
     logical_appeal               0.0000        0.5333    53.33
ms   authority_endorsement        0.5000        0.6429    14.29
     evidence_based               0.4091        0.4412     3.21
     logical_appeal               0.5294        0.6923    16.29
ru   authority_endorsement        0.0000        0.5000    50.00
     evidence_based               0.4444        0.4000    -4.44
     logical_appeal               0.5000        0.4667    -3.33
sr   authority_endorsement        0.3333        0.6000    26.67
     evidence_based               0.8125        0.7143    -9.82
     logical_appeal               0.5294        0.6250     9.56
th   authority_endorsement        0.0000        0.3333    33.33
     evidence_based               0.6250        0.3600   -26.50
     logical_appeal               0.2000        0.4000    20.00
vi   authority_endorsement        0.0000        0.4000    40.00
     evidence_based               0.4167        0.4211     0.44
     logical_appeal               0.5000        0.5714     7.14
zh   authority_endorsement        0.5000        0.7143    21.43
     evidence_based               0.4667        0.5714    10.48
     logical_appeal               0.6154        0.6800     6.46

In [11]:
# Aggregate per technique across all langs.
tech_pooled = (
    ok.groupby(['technique', 'quadrant'])['complied']
      .mean().unstack('quadrant')
)
tech_pooled['diff_pp'] = ((tech_pooled['trigger_only'] - tech_pooled['concept_deep']) * 100).round(2)
tech_pooled.round(4)

quadrant,concept_deep,trigger_only,diff_pp
technique,,,
authority_endorsement,0.3158,0.5361,22.03
evidence_based,0.4779,0.5152,3.72
logical_appeal,0.4957,0.6000,10.43


## 2. Paraphrase attack (Exp 2, across models)

Same idea: filter to `rewrite_status == "ok"` and recompute Qwen ASR − Llama ASR.

In [12]:
frames = []
for p in sorted(SHARED.glob('responses_eval-*.json')):
    with open(p) as f:
        d = json.load(f)
    frames.append(pd.DataFrame(d['responses']))
para = pd.concat(frames, ignore_index=True)
print(f'total rows: {len(para):,}')
print(para['rewrite_status'].value_counts())

# Same rewrite is reused across both models, so rewriter-refusal status is
# symmetric by construction. Sanity-check anyway.
rew_para = (
    para.assign(rewriter_refused=lambda d: d['rewrite_status'] == 'rewriter_refused')
        .groupby(['lang', 'evaluated_on'])['rewriter_refused']
        .mean().unstack('evaluated_on').round(3)
)
rew_para

total rows: 1,200
rewrite_status
ok                  976
rewriter_refused    224
Name: count, dtype: int64


evaluated_on,llama,qwen
lang,,
ar,0.147,0.147
en,0.207,0.207
ru,0.173,0.173
zh,0.220,0.220


In [13]:
ok_para = para[para['rewrite_status'] == 'ok']
print(f'ok rows: {len(ok_para):,}  ({len(ok_para)/len(para):.1%} of total)')

asr = (
    ok_para.groupby(['lang', 'evaluated_on'])
           .agg(n=('refused', 'size'),
                asr=('complied', 'mean'))
           .reset_index()
)
asr_pivot = asr.pivot(index='lang', columns='evaluated_on', values='asr')
n_para    = asr.pivot(index='lang', columns='evaluated_on', values='n')
asr_pivot['diff_pp']  = ((asr_pivot.get('qwen', 0) - asr_pivot.get('llama', 0)) * 100).round(2)
asr_pivot['n_llama']  = n_para['llama']
asr_pivot['n_qwen']   = n_para['qwen']
asr_pivot = asr_pivot.round({'llama': 4, 'qwen': 4})
asr_pivot.loc['__mean__'] = [asr_pivot['llama'].mean(),
                              asr_pivot['qwen'].mean(),
                              asr_pivot['diff_pp'].mean(),
                              asr_pivot['n_llama'].sum(),
                              asr_pivot['n_qwen'].sum()]
asr_pivot

ok rows: 976  (81.3% of total)


evaluated_on,llama,qwen,diff_pp,n_llama,n_qwen
lang,,,,,
ar,0.468800,0.882800,41.410,128.0,128.0
en,0.277300,0.882400,60.500,119.0,119.0
ru,0.564500,0.967700,40.320,124.0,124.0
zh,0.410300,0.940200,52.990,117.0,117.0
__mean__,0.430225,0.918275,48.805,488.0,488.0


In [14]:
# Per (lang, strategy) breakdown on the ok subset.
para_tech = (
    ok_para.groupby(['lang', 'strategy', 'evaluated_on'])['complied']
           .mean().unstack('evaluated_on')
)
para_tech['diff_pp'] = ((para_tech['qwen'] - para_tech['llama']) * 100).round(2)
para_tech.round(4)

evaluated_on                llama    qwen  diff_pp
lang strategy                                     
ar   academic_framing      0.3684  0.9211    55.26
     hypothetical_framing  0.2727  0.8409    56.82
     indirect_query        0.7391  0.8913    15.22
en   academic_framing      0.2821  0.8974    61.54
     hypothetical_framing  0.1463  0.8049    65.85
     indirect_query        0.4103  0.9487    53.85
ru   academic_framing      0.4865  1.0000    51.35
     hypothetical_framing  0.3636  0.9318    56.82
     indirect_query        0.8372  0.9767    13.95
zh   academic_framing      0.2286  0.9714    74.29
     hypothetical_framing  0.3000  0.8500    55.00
     indirect_query        0.6667  1.0000    33.33

## 3. Sign test — how many languages move in the predicted direction?

Magnitude can be muted by the OK-subset shrinkage, but the sign test is unaffected. 12/12 with p = 0.00024 is the headline you can quote for Exp 1; 4/4 (p = 0.0625, small n) for Exp 2.

In [15]:
from scipy.stats import binomtest

# Exp 1: positive diff_pp counts.
diffs_exp1 = comp_pivot.drop('__mean__', errors='ignore')['diff_pp']
n_exp1, pos_exp1 = len(diffs_exp1), int((diffs_exp1 > 0).sum())
print(f'Exp 1 (persuasion): {pos_exp1}/{n_exp1} languages with trigger_only > concept_deep  '
      f'(sign test p = {binomtest(pos_exp1, n_exp1, p=0.5, alternative="greater").pvalue:.4f})')

# Exp 2: positive diff_pp counts.
diffs_exp2 = asr_pivot.drop('__mean__', errors='ignore')['diff_pp']
n_exp2, pos_exp2 = len(diffs_exp2), int((diffs_exp2 > 0).sum())
print(f'Exp 2 (paraphrase): {pos_exp2}/{n_exp2} languages with Qwen > Llama  '
      f'(sign test p = {binomtest(pos_exp2, n_exp2, p=0.5, alternative="greater").pvalue:.4f})')

Exp 1 (persuasion): 11/12 languages with trigger_only > concept_deep  (sign test p = 0.0032)
Exp 2 (paraphrase): 4/4 languages with Qwen > Llama  (sign test p = 0.0625)
